In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc_context
import matplotlib.patheffects as path_effects
from matplotlib.collections import LineCollection
from matplotlib import patches
from matplotlib.cm import ScalarMappable
from matplotlib import ticker
import sunpy
import sunpy.map
from sunpy.coordinates import propagate_with_solar_surface
from sunpy.coordinates.spice import get_rotation_matrix
import astropy
from astropy.coordinates import SkyCoord
import astropy.units as u
import astropy.constants as const
from astropy.io import fits, ascii
from astropy.time import Time
from astropy.convolution import convolve, Gaussian2DKernel
from astropy.wcs import WCS
from astropy.visualization import ImageNormalize, AsinhStretch
from streamtracer import StreamTracer, VectorGrid
from extrapolater import PotentialField
from helpers import from_local
from ndcube.wcs.tools import unwrap_wcs_to_fitswcs
import pyvista as pv

import h5py 
import dask.array as da 
from ndcube import NDCube
from fancy_colorbar import plot_colorbar
import os 
os.environ["SPICE_KERNEL_PATH"] = "/cluster/home/zhuyin/scripts/spice_kernel/"
from mag_reproject import hgs_local_to_heeq_cart

from copy import deepcopy

import sys
sys.path.append("/cluster/home/zhuyin/scripts/MHSXtraPy/")

from mhsxtrapy.b3d import WhichSolution
from mhsxtrapy.examples import multipole
from mhsxtrapy.field2d import Field2dData, FluxBalanceState, check_fluxbalance
from mhsxtrapy.field3d import calculate_magfield, Field3dData
from mhsxtrapy.plotting.vis import (
    plot_ddensity_xy,
    plot_ddensity_z,
    plot_dpressure_xy,
    plot_dpressure_z,
    plot_magnetogram_2D,
    plot_magnetogram_3D,
)

from IPython.display import HTML, display

In [3]:
data3d = Field3dData.load("../../data/pid_1_123_aux/MHSXtra_results/SOTSP_test_full_v2/")

In [5]:
data3d.field.shape

(1072, 1920, 536, 3)

In [8]:
nx, ny, nz, nf = 960, 536, 536, 536

pixelsize_x = 0.23712652199468398
pixelsize_y = pixelsize_x
pixelsize_z = pixelsize_x

# x_arr = np.linspace(xmin, xmax, nx, dtype=np.float64)
# y_arr = np.linspace(ymin, ymax, ny, dtype=np.float64)
# z_arr = np.linspace(zmin, zmax, nz, dtype=np.float64)
x_arr = np.arange(nx) * pixelsize_x
y_arr = np.arange(ny) * pixelsize_y
z_arr = np.arange(nz) * pixelsize_z

bx_extra = data3d.field[:,:,:,1]
by_extra = data3d.field[:,:,:,0]
bz_extra = data3d.field[:,:,:,2]

bx_extra = bx_extra[ny:ny*2, nx:nx*2,:].transpose(1,0,2)
by_extra = by_extra[ny:ny*2, nx:nx*2,:].transpose(1,0,2)
bz_extra = bz_extra[ny:ny*2, nx:nx*2,:].transpose(1,0,2)

In [9]:
with h5py.File("../../data/pid_1_123_aux/MHSXtra_results/Antoine/SOTSP_MHS_20221024.h5", "w") as f:
    f.create_dataset("bx", data=bx_extra)
    f.create_dataset("by", data=by_extra)
    f.create_dataset("bz", data=bz_extra)